<a href="https://colab.research.google.com/github/vkawao/OffsideDetection/blob/main/OffsideDetection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **READ ME!**

If you are viewing this in GitHub, there seems to be an issue if you click anything on the code. Please press the **"Open in Colab"** button above to run the code to avoid any issues.

---------------------------------------------------------------

There are two main code blocks in this notebook:

# **1️⃣ Setup Block**
Run this block first to download and import the necessary libraries

# **2️⃣ cVAR: Computer Vision Assistant Referee Block**
After setup, run this block to start the interactive program. You are encouraged to go into Full Screen Output by pressing the ... on the left of the output terminal.

---------------------------------------------------------------

# **⚽ Test Images**
You can download images for testing the program [by clicking here!](https://drive.google.com/drive/folders/1uP0b5ujFsJ-yWu0iJ1jCut_gUeBggxHK?usp=drive_link)

---------------------------------------------------------------

# **How to use**
1. Run the program
2. Upload an image of a potential Offside violation **(Make sure you know which team is attacking, and which side they are attacking first!)**
3. Adjust HSV parameters so that Team A and Team B are separated clearly, and referees are not a part of a team
4. Adjust Match Configuration (Who's attacking, which direction)
5. Click Confirm
6. Offside Decision!

---------------------------------------------------------------

Suuuui... ⛹️‍♂️⛹️⛹️‍♀️

# **SETUP**

In [ ]:
!pip install ultralytics
import cv2
import random
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ultralytics import YOLO
from google.colab import files
from google.colab import output
from IPython.display import display, clear_output
output.clear()

# **cVAR:** *Computer Vision Assistant Referee*

In [ ]:
# =============================================================================
#                                FUNCTIONS
# =============================================================================
def get_intersection(line1, line2):
    x1, y1, x2, y2 = line1
    x3, y3, x4, y4 = line2

    den = (x1 - x2) * (y3 - y4) - (y1 - y2) * (x3 - x4)
    if den == 0:
        return None

    px = ((x1*y2 - y1*x2)*(x3 - x4) - (x1 - x2)*(x3*y4 - y3*x4)) / den
    py = ((x1*y2 - y1*x2)*(y3 - y4) - (y1 - y2)*(x3*y4 - y3*x4)) / den
    return (int(px), int(py))

def get_distance(pt, line):
    px, py = pt
    x1, y1, x2, y2 = line

    den = np.sqrt((y2 - y1)**2 + (x2 - x1)**2)
    if den == 0:
        return 9999

    num = abs((y2 - y1)*px - (x2 - x1)*py + x2*y1 - y2*x1)
    return num / den

def classify_team(med_h, med_s, med_v, v_thresh, s_thresh, hue_a, hue_b):
    # Filter out dark colors (Managers, Refs, Lines)
    if med_v < v_thresh or med_s < s_thresh:
        return 'other'

    # Team A checks
    if hue_a[0] <= med_h <= hue_a[1]:
        return 'team_a'

    # Team B checks
    if hue_b[0] <= med_h <= hue_b[1]:
        return 'team_b'

    return 'other'

def get_color_from_hue_range(hue_range, is_rgb=False):
    # Find the midpoint of the selected hue range
    mid_h = (hue_range[0] + hue_range[1]) // 2

    # Create a 1x1 pixel with that Hue, and max Saturation/Value for visibility
    hsv_pixel = np.array([[[mid_h, 255, 255]]], dtype=np.uint8)

    if is_rgb:
        rgb_pixel = cv2.cvtColor(hsv_pixel, cv2.COLOR_HSV2RGB)
        return (int(rgb_pixel[0][0][0]), int(rgb_pixel[0][0][1]), int(rgb_pixel[0][0][2]))
    else:
        bgr_pixel = cv2.cvtColor(hsv_pixel, cv2.COLOR_HSV2BGR)
        return (int(bgr_pixel[0][0][0]), int(bgr_pixel[0][0][1]), int(bgr_pixel[0][0][2]))

# =============================================================================
#                              VAR SYSTEM CLASSES
# =============================================================================
class VARSystem:
    def __init__(self):
        self.model = YOLO('yolov8m.pt', verbose=False)
        self.img = None
        self.img_rgb = None
        self.field_mask = None
        self.field_only = None
        self.s_channel = None
        self.field_edges = None
        self.line_img = None
        self.cached_players = []
        self.best_vp = None
        self.valid_lines = []

    def load_and_process_image(self, filename):
        # 1. Load Image
        self.img = cv2.imread(filename)
        self.img = cv2.resize(self.img, (1280, 720))
        self.img_rgb = cv2.cvtColor(self.img, cv2.COLOR_BGR2RGB)

        # 2. Extract Pitch Mask
        hsv_img = cv2.cvtColor(self.img, cv2.COLOR_BGR2HSV)
        lower_green = np.array([30, 40, 40])
        upper_green = np.array([85, 255, 255])
        self.field_mask = cv2.inRange(hsv_img, lower_green, upper_green)

        kernel = np.ones((5,5), np.uint8)
        self.field_mask = cv2.morphologyEx(self.field_mask, cv2.MORPH_CLOSE, kernel)
        self.field_mask = cv2.morphologyEx(self.field_mask, cv2.MORPH_OPEN, kernel)

        # 3. Calculate Vanishing Point (RANSAC)
        self._calculate_vanishing_point()

        # 4. Run YOLO and Cache Players
        self._extract_players()

    def _calculate_vanishing_point(self):
        self.field_only = cv2.bitwise_and(self.img, self.img, mask=self.field_mask)
        _, self.s_channel, _ = cv2.split(self.field_only)

        canny_edges = cv2.Canny(self.s_channel, 10, 15)
        mask_dilated = cv2.dilate(self.field_mask, np.ones((15,15), np.uint8))
        self.field_edges = cv2.bitwise_and(canny_edges, canny_edges, mask=mask_dilated)

        lines = cv2.HoughLinesP(self.field_edges, 1, np.pi/180, 150, minLineLength=100, maxLineGap=50)
        self.valid_lines = []
        self.line_img = self.img.copy()

        if lines is not None:
            for line in lines:
                x1, y1, x2, y2 = line[0]
                angle = np.abs(np.arctan2(y2 - y1, x2 - x1) * 180.0 / np.pi)

                if 20 < angle < 160 and angle != 90:
                    self.valid_lines.append((x1, y1, x2, y2))
                    cv2.line(self.line_img, (x1, y1), (x2, y2), (0, 0, 255), 2)

        max_inliers = -1
        self.best_vp = None

        # RANSAC
        if len(self.valid_lines) >= 2:
            for _ in range(200):
                l1, l2 = random.sample(self.valid_lines, 2)
                vp = get_intersection(l1, l2)

                if vp is None:
                    continue

                inliers = 0
                for line in self.valid_lines:
                    if get_distance(vp, line) < 15.0:
                        inliers += 1

                if inliers > max_inliers:
                    max_inliers = inliers
                    self.best_vp = vp

    def _extract_players(self):
        contours, _ = cv2.findContours(self.field_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        solid_pitch_mask = np.zeros_like(self.field_mask)

        if contours:
            largest_contour = max(contours, key=cv2.contourArea)
            cv2.drawContours(solid_pitch_mask, [largest_contour], -1, 255, thickness=cv2.FILLED)

        ai_input_img = cv2.bitwise_and(self.img, self.img, mask=solid_pitch_mask)
        results = self.model(ai_input_img, classes=[0], conf=0.3, imgsz=1280, verbose=False)
        self.cached_players = []

        for result in results:
            for box in result.boxes:
                x1, y1, x2, y2 = [int(val) for val in box.xyxy[0]]
                w = x2 - x1
                h = y2 - y1

                # Core Torso Crop
                cy1 = int(y1 + h * 0.15)
                cy2 = int(y1 + h * 0.45)
                cx1 = int(x1 + w * 0.3)
                cx2 = int(x1 + w * 0.7)

                chest_roi = self.img[cy1:cy2, cx1:cx2]

                if chest_roi.size > 0:
                    chest_hsv = cv2.cvtColor(chest_roi, cv2.COLOR_BGR2HSV)
                    med_h = np.median(chest_hsv[:,:,0])
                    med_s = np.median(chest_hsv[:,:,1])
                    med_v = np.median(chest_hsv[:,:,2])

                    self.cached_players.append({
                        'coords': (x1, y1, w, h),
                        'hsv': (med_h, med_s, med_v)
                    })

    def get_bottom_x(self, feet_x, feet_y): # Get feet
        vp_x = self.best_vp[0]
        vp_y = self.best_vp[1]
        img_height = self.img.shape[0]

        if feet_y == vp_y:
            return feet_x

        return vp_x + (img_height - vp_y) * (feet_x - vp_x) / (feet_y - vp_y)


# =============================================================================
#                               UI LAUNCHER SCRIPT
# =============================================================================
print("Please upload a match image (.jpg or .png):")
uploaded = files.upload()
filename = next(iter(uploaded))

print("\nInitializing System (Please wait...)")
var_app = VARSystem()
var_app.load_and_process_image(filename)

clear_output()

# --- Configured globally for Team A and Team B ---
team_dropdown = widgets.Dropdown(options=['Team A', 'Team B'], value='Team A', description='Attacking:')
dir_dropdown = widgets.Dropdown(options=['Left', 'Right'], value='Right', description='Direction:')
v_thresh_slider = widgets.IntSlider(value=40, min=0, max=100, step=1, description='Black (V):')
s_thresh_slider = widgets.IntSlider(value=30, min=0, max=100, step=1, description='White (S):')
hue_a_range = widgets.IntRangeSlider(value=[0, 28], min=0, max=179, step=1, description='Team A Hue:')
hue_b_range = widgets.IntRangeSlider(value=[40, 135], min=0, max=179, step=1, description='Team B Hue:')
run_button = widgets.Button(description="Confirm & Run VAR", button_style='success', icon='check')

result_output = widgets.Output()

controls_box = widgets.VBox([
    widgets.HTML("<b>Match Configuration</b>"),
    team_dropdown,
    dir_dropdown,
    widgets.HTML("<br><b>HSV Tuning Parameters</b>"),
    v_thresh_slider,
    s_thresh_slider,
    hue_a_range,
    hue_b_range,
    widgets.HTML("<hr>"),
    run_button
], layout=widgets.Layout(width='320px', padding='15px', border='2px solid #e0e0e0', border_radius='10px'))

# Interactive Live Preview
# Interactive Live Preview
# Interactive Live Preview
def update_live_preview(team, direction, v_thresh, s_thresh, hue_a, hue_b):
    display_img = var_app.img_rgb.copy()
    team_a_count = 0
    team_b_count = 0

    # Dynamically generate RGB colors from the sliders
    color_a = get_color_from_hue_range(hue_a, is_rgb=True)
    color_b = get_color_from_hue_range(hue_b, is_rgb=True)

    # Variables to hold our representative player crops
    team_a_crop = None
    team_b_crop = None

    for p in var_app.cached_players:
        x1, y1, w, h = p['coords']
        med_h, med_s, med_v = p['hsv']

        team_class = classify_team(med_h, med_s, med_v, v_thresh, s_thresh, hue_a, hue_b)

        if team_class == 'team_a':
            color = color_a
            team_a_count += 1
            # Capture the first Team A player for the legend (with a 5px padding)
            if team_a_crop is None:
                px1, py1 = max(0, x1-5), max(0, y1-5)
                px2, py2 = min(var_app.img.shape[1], x1+w+5), min(var_app.img.shape[0], y1+h+5)
                team_a_crop = var_app.img_rgb[py1:py2, px1:px2]

        elif team_class == 'team_b':
            color = color_b
            team_b_count += 1
            # Capture the first Team B player for the legend (with a 5px padding)
            if team_b_crop is None:
                px1, py1 = max(0, x1-5), max(0, y1-5)
                px2, py2 = min(var_app.img.shape[1], x1+w+5), min(var_app.img.shape[0], y1+h+5)
                team_b_crop = var_app.img_rgb[py1:py2, px1:px2]
        else:
            color = (255, 255, 255) # White for others

        # Draw main image boxes
        cv2.rectangle(display_img, (x1, y1), (x1+w, y1+h), color, 2)
        cv2.circle(display_img, (int(x1 + w/2), y1+h), 5, (0, 255, 0), -1)

        hsv_text = f"H:{int(med_h)} S:{int(med_s)} V:{int(med_v)}"
        cv2.putText(display_img, hsv_text, (x1, max(15, y1 - 8)), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 0, 0), 2, cv2.LINE_AA)
        cv2.putText(display_img, hsv_text, (x1, max(15, y1 - 8)), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 255), 1, cv2.LINE_AA)

    # --- NEW ADVANCED PLOTTING LAYOUT ---
    fig = plt.figure(figsize=(10, 8))
    gs = fig.add_gridspec(3, 2) # 3 rows, 2 columns layout

    # 1. Main Preview (Spans the top 2 rows)
    ax_main = fig.add_subplot(gs[0:2, :])
    ax_main.imshow(display_img)
    ax_main.set_title(f"LIVE PREVIEW | Team A: {team_a_count} | Team B: {team_b_count} \n{team} attacking {direction}")
    ax_main.axis('off')

    # 2. Team A Representative (Bottom Left)
    ax_a = fig.add_subplot(gs[2, 0])
    if team_a_crop is not None:
        ax_a.imshow(team_a_crop)
        ax_a.set_title("Team A", color=[c/255 for c in color_a], fontweight='bold')
        ax_a.set_xticks([]) # Remove grid ticks but keep the border
        ax_a.set_yticks([])
        for spine in ax_a.spines.values(): # Thicken the border and match the dynamic color
            spine.set_edgecolor([c/255 for c in color_a])
            spine.set_linewidth(4)
    else:
        ax_a.text(0.5, 0.5, "No Team A Detected", ha='center', va='center')
        ax_a.axis('off')

    # 3. Team B Representative (Bottom Right)
    ax_b = fig.add_subplot(gs[2, 1])
    if team_b_crop is not None:
        ax_b.imshow(team_b_crop)
        ax_b.set_title("Team B", color=[c/255 for c in color_b], fontweight='bold')
        ax_b.set_xticks([])
        ax_b.set_yticks([])
        for spine in ax_b.spines.values():
            spine.set_edgecolor([c/255 for c in color_b])
            spine.set_linewidth(4)
    else:
        ax_b.text(0.5, 0.5, "No Team B Detected", ha='center', va='center')
        ax_b.axis('off')

    plt.tight_layout()
    plt.show()

preview_output = widgets.interactive_output(update_live_preview, {
    'team': team_dropdown,
    'direction': dir_dropdown,
    'v_thresh': v_thresh_slider,
    's_thresh': s_thresh_slider,
    'hue_a': hue_a_range,
    'hue_b': hue_b_range
})

display(widgets.HBox([controls_box, preview_output]), result_output)

# Begin Offside Detection
def on_confirm(b):
    with result_output:
        clear_output(wait=True)
        print("Executing VAR Pipeline...\n")

        if var_app.best_vp is None:
            print("Error: No Vanishing Point found.")
            return

        team_a_list = []
        team_b_list = []

        yolo_boxes_img = var_app.img.copy()
        player_lines_img = var_app.img.copy()

        # Dynamically generate BGR colors from the sliders
        color_a = get_color_from_hue_range(hue_a_range.value, is_rgb=False)
        color_b = get_color_from_hue_range(hue_b_range.value, is_rgb=False)

        for p in var_app.cached_players:
            x1, y1, w, h = p['coords']
            med_h, med_s, med_v = p['hsv']

            team_class = classify_team(
                med_h, med_s, med_v,
                v_thresh_slider.value,
                s_thresh_slider.value,
                hue_a_range.value,
                hue_b_range.value
            )

            if team_class == 'team_a':
                team_a_list.append((x1, y1, w, h))
                box_color = color_a
            elif team_class == 'team_b':
                team_b_list.append((x1, y1, w, h))
                box_color = color_b
            else:
                box_color = (255, 255, 255) # White

            cv2.rectangle(yolo_boxes_img, (x1, y1), (x1+w, y1+h), box_color, 2)

            feet_x = int(x1 + w/2)
            feet_y = int(y1 + h)

            cv2.rectangle(player_lines_img, (x1, y1), (x1+w, y1+h), box_color, 2)
            cv2.circle(player_lines_img, (feet_x, feet_y), 5, (0, 255, 0), -1)

            if var_app.best_vp:
                bottom_x = var_app.get_bottom_x(feet_x, feet_y)
                cv2.line(player_lines_img, var_app.best_vp, (int(bottom_x), var_app.img.shape[0]), box_color, 1)

        # Dynamic logic based on Team A vs Team B
        if team_dropdown.value == 'Team A':
            attackers = team_a_list
            defenders = team_b_list
        else:
            attackers = team_b_list
            defenders = team_a_list

        if len(defenders) == 0:
            print("Error: No defending players detected.")
            return

        last_defender = None

        if dir_dropdown.value == 'Right':
            extreme_def_x = -999999
        else:
            extreme_def_x = 999999

        for (x, y, bw, bh) in defenders: # Check furthest back defender
            feet_x = int(x + bw/2)
            feet_y = int(y + bh)
            bottom_x = var_app.get_bottom_x(feet_x, feet_y)

            if dir_dropdown.value == 'Right' and bottom_x > extreme_def_x: # If attacking right
                extreme_def_x = bottom_x
                last_defender = (feet_x, feet_y)
            elif dir_dropdown.value == 'Left' and bottom_x < extreme_def_x: # If attacking left
                extreme_def_x = bottom_x
                last_defender = (feet_x, feet_y)

        final_img = var_app.img.copy()
        img_height = final_img.shape[0]

        cv2.line(final_img, var_app.best_vp, (int(extreme_def_x), img_height), (0, 0, 255), 3)
        cv2.circle(final_img, last_defender, 8, (0, 0, 255), -1)

        text_x = int(extreme_def_x) - 150 if dir_dropdown.value == 'Right' else int(extreme_def_x) + 10
        cv2.putText(final_img, "OFFSIDE LINE", (text_x, img_height - 20), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)

        offside_count = 0
        for (x, y, bw, bh) in attackers: # Check deepest attacker
            feet_x = int(x + bw/2)
            feet_y = int(y + bh)
            bottom_x = var_app.get_bottom_x(feet_x, feet_y)

            is_offside = False
            if dir_dropdown.value == 'Right' and bottom_x > extreme_def_x: # If attacking right
                is_offside = True
            elif dir_dropdown.value == 'Left' and bottom_x < extreme_def_x: # If attacking left
                is_offside = True

            if is_offside:
                offside_count += 1
                cv2.line(final_img, var_app.best_vp, (int(bottom_x), img_height), (0, 165, 255), 2)
                cv2.rectangle(final_img, (x, y), (x+bw, y+bh), (0, 0, 255), 3)
                cv2.putText(final_img, "OFFSIDE!", (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

        print(f"DECISION: Found {offside_count} player(s) in an offside position.")
        plt.figure(figsize=(10, 6))
        plt.imshow(cv2.cvtColor(final_img, cv2.COLOR_BGR2RGB))
        plt.title(f"VAR Decision: {team_dropdown.value} attacking {dir_dropdown.value}")
        plt.axis('off')
        plt.show()

# =============================================================================
#                               PIPELINE DISPLAY
# =============================================================================

        print("\n" + "="*50)
        print("BACKGROUND PIPELINE PROCESSES")
        print("="*50)

        # 1. Field Segmentation Images
        print("\n--- OUTPUT 1: Field Segmentation ---")
        fig1, axes1 = plt.subplots(1, 3, figsize=(18, 6))

        axes1[0].imshow(var_app.img_rgb)
        axes1[0].set_title("1. Original Image")
        axes1[0].axis('off')

        axes1[1].imshow(var_app.field_mask, cmap='gray')
        axes1[1].set_title("2. Tuned HSV Mask")
        axes1[1].axis('off')

        axes1[2].imshow(cv2.cvtColor(var_app.field_only, cv2.COLOR_BGR2RGB))
        axes1[2].set_title("3. Segmented Field")
        axes1[2].axis('off')

        plt.tight_layout()
        plt.show()

        # 2. Line Detection Images
        print("\n--- OUTPUT 2: Line Detection ---")
        fig2, axes2 = plt.subplots(1, 3, figsize=(18, 6))

        axes2[0].imshow(var_app.s_channel, cmap='gray')
        axes2[0].set_title("1. Saturation Channel")
        axes2[0].axis('off')

        axes2[1].imshow(var_app.field_edges, cmap='gray')
        axes2[1].set_title("2. Filtered Field Edges")
        axes2[1].axis('off')

        axes2[2].imshow(cv2.cvtColor(var_app.line_img, cv2.COLOR_BGR2RGB))
        axes2[2].set_title("3. Hough Lines Detected")
        axes2[2].axis('off')

        plt.tight_layout()
        plt.show()

        # 3. Geometry
        print("\n--- OUTPUT 3: 3D Geometry ---")
        plt.figure(figsize=(10, 6))
        plt.imshow(var_app.img_rgb)
        if var_app.best_vp:
            plt.scatter(var_app.best_vp[0], var_app.best_vp[1], color='red', s=150, zorder=5)
            for l in var_app.valid_lines:
                plt.plot([l[0], var_app.best_vp[0]], [l[1], var_app.best_vp[1]], color='blue', linewidth=1)
            plt.xlim(min(-500, var_app.best_vp[0] - 200), max(var_app.img.shape[1] + 500, var_app.best_vp[0] + 200))
            plt.ylim(var_app.img.shape[0], min(-500, var_app.best_vp[1] - 200))
        plt.title("3D Geometry: Vanishing Point")
        plt.axis('off')
        plt.show()

        # 4. Player Detection & Projections
        print("\n--- OUTPUT 4: Player Detection ---")
        fig4, axes4 = plt.subplots(1, 2, figsize=(18, 6))

        axes4[0].imshow(cv2.cvtColor(yolo_boxes_img, cv2.COLOR_BGR2RGB))
        axes4[0].set_title("Player Detection YOLO")
        axes4[0].axis('off')

        axes4[1].imshow(cv2.cvtColor(player_lines_img, cv2.COLOR_BGR2RGB))
        axes4[1].set_title("Vanishing Point Projections")
        axes4[1].axis('off')

        plt.tight_layout()
        plt.show()

run_button.on_click(on_confirm)

Please upload a match image (.jpg or .png):
